# Layer 2 of 4 - Character n-gram reference

Consumes the persisted data splits from Layer 1. It tunes the smoothed n-gram control on validation BPC and saves its selected settings for the final comparison.


In [ ]:
import json
from collections import Counter

import numpy as np

from pathlib import Path

current = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (current, *current.parents) if (path / ".git").exists()), current)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RUNS_DIR = RESULTS_DIR / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
for directory in (DATA_DIR, RESULTS_DIR, RUNS_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
DATA_PATHS = {name: DATA_DIR / f"{name}.txt" for name in ("train", "val", "test")}
missing = [path for path in DATA_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Run layers/01_data/01_data_preparation.ipynb first; missing: " + ", ".join(map(str, missing)))
train_text, val_text, test_text = (DATA_PATHS[name].read_text(encoding="utf-8") for name in ("train", "val", "test"))
VOCAB_SIZE = len(set(train_text))
EVAL_START = 256


## 4. N-gram

A character n-gram with interpolated absolute-discounting smoothing. For a history *h* and next character *c*:

P_n(c | h) = max(count(h, c) − d, 0) / count(h) + d · types(h) / count(h) · P_{n−1}(c | shorter h)

types(h) is the number of distinct characters seen after *h*. Histories never seen in training fall back fully to the shorter history. The unigram level uses add-one smoothing, so no character gets probability zero.

In [ ]:
class NgramModel:
    def __init__(self, text, max_order):
        self.max_order = max_order
        self.unigram_counts = Counter(text)
        self.total_chars = len(text)
        self.ngram_counts, self.history_counts, self.history_types = {}, {}, {}
        for order in range(2, max_order + 1):
            ngrams = Counter(text[i:i + order] for i in range(len(text) - order + 1))
            history_counts, history_types = Counter(), Counter()
            for ngram, count in ngrams.items():
                history_counts[ngram[:-1]] += count
                history_types[ngram[:-1]] += 1
            self.ngram_counts[order] = ngrams
            self.history_counts[order] = history_counts
            self.history_types[order] = history_types

    def probability_by_order(self, history, char, discount):
        probability = (self.unigram_counts[char] + 1) / (self.total_chars + VOCAB_SIZE)
        probabilities = [probability]
        for order in range(2, self.max_order + 1):
            context = history[-(order - 1):]
            history_count = self.history_counts[order][context]
            if history_count:
                discounted = max(self.ngram_counts[order][context + char] - discount, 0) / history_count
                backoff_weight = discount * self.history_types[order][context] / history_count
                probability = discounted + backoff_weight * probability
            probabilities.append(probability)
        return probabilities

## 5. Metric: fixed-target bits-per-character

Every model scores the same targets: every character from position 257 of a split onward. A model with context length L predicts each target from the L characters before it, and only the last position of each window is scored. BPC = cross-entropy in nats / ln 2.

In [ ]:
def ngram_bpc_by_order(model, split_text, discount):
    bits = np.zeros(model.max_order)
    for t in range(EVAL_START, len(split_text)):
        history = split_text[t - model.max_order + 1:t]
        bits -= np.log2(model.probability_by_order(history, split_text[t], discount))
    return bits / (len(split_text) - EVAL_START)


## 6. Reference condition 1: n-gram

Order (1–8) and discount *d* are chosen by validation BPC. The model is deterministic, so it needs only one run.

In [ ]:
ngram = NgramModel(train_text, max_order=8)

ngram_grid = []
for discount in [0.5, 0.75, 0.9]:
    for order, bpc in enumerate(ngram_bpc_by_order(ngram, val_text, discount), start=1):
        ngram_grid.append({"order": order, "discount": discount, "val_bpc": float(bpc)})

best_ngram = min(ngram_grid, key=lambda row: row["val_bpc"])
(RESULTS_DIR / "ngram.json").write_text(json.dumps({"best": best_ngram, "grid": ngram_grid}, indent=2))
best_ngram